# Tweety-7a : Frameworks d'Argumentation Etendus (C#)

**Navigation** : [<< 6-Structured-Argumentation](Tweety-6-Structured-Argumentation.ipynb) | [Index](README.md) | [7b-Ranking-Probabilistic >>](Tweety-7b-Ranking-Probabilistic.ipynb)

## Frameworks d'Argumentation Etendus (jumeau .NET)

Ce notebook est le **jumeau C# (.NET Interactive)** de [Tweety-7a-Extended-Frameworks.ipynb](Tweety-7a-Extended-Frameworks.ipynb). Au-dela du cadre abstrait de Dung (Tweety-5), il explore les **frameworks etendus** : ADF (conditions d'acceptation), SetAF (attaques ensembles), EAF (attaques sur les attaques), VAF (argumentation value-based).

### Value-add du jumeau C# (Prong B, EPIC #3801)

Le twin Python s'appuie sur **jpype + la JVM Tweety** (`org.tweetyproject.arg.adf`, `.bipolar`, `.setaf`, `.eaf`, ...) pour evaluer les semantiques. Ce jumeau reimplémente tout **from-scratch en C# / BCL .NET 9** :

| Pilier | Python (Tweety JVM via jpype) | C# (from-scratch) |
|--------|-------------------------------|-------------------|
| **ADF** (Abstract Dialectical Framework) | `AbstractDialecticalFramework` + solveur SAT NativeMinisat | conditions d'acceptation comme `Func<>`, **logique 3-valued de Kleene**, fixed-point grounded |
| **SetAF** (Nielsen-Parsons 2011) | `SetAttack` Tweety | attaques `HashSet<string> -> string`, labelling grounded (attaque active ssi tous attaquants IN) |
| **EAF** (Modgil, attaques sur attaques) | `RecursiveAttack` Tweety | meta-attaques `(c,(a,b))`, revocateur d'attaque -> reinstatement |
| **VAF** (Bench-Capon, value-based) | raisonneur Tweety | ordre sur les valeurs -> defeat conditionnel |

### Objectifs d'apprentissage
- Comprendre comment les ADF generalisent Dung via des **conditions d'acceptation**.
- Maitriser les attaques d'**ensembles** (SetAF) et les **meta-attaques** (EAF).
- Voir comment les **valeurs** (VAF) rendent la defaite conditionnelle a un ordre de preference.

### Duree estimee : 55 minutes


## 1. Pourquoi etendre le cadre de Dung ?

Le cadre abstrait de Dung (1995, voir [Tweety-5](Tweety-5-Abstract-Argumentation.ipynb)) modelise l'attaque binaire `a -> b` entre arguments entiers. Mais la realite argumentative est plus riche :

- **Conditions d'acceptation** : la validite de `a` peut dependre d'une formule logique sur d'autres arguments (ADF).
- **Attaques collectives** : un *ensemble* d'arguments peut etre requis pour en attaquer un (SetAF).
- **Attaques d'attaques** : un argument peut attaquer une *attaque* elle-meme, la revoquant (EAF, Modgil 2010).
- **Valeurs** : chaque argument promeut une valeur ; la defaite depend d'un ordre de valeurs (VAF, Bench-Capon 2003).

Nous implementons ces 4 extensions **from-scratch**, avec a chaque fois un exemple **verifiable analytiquement** (re-derivation a la main de l'extension grounded).


## 2. Brique commune : logique 3-valued de Kleene

Les ADF utilisent une logique a **trois valeurs** : Vrai (T), Faux (F), Indetermine (U). La negation, conjonction, disjonction suivent les tables de **Kleene** (U propage l'incertitude) :

| `a` | `b` | `a AND b` | `a OR b` | `NOT a` |
|-----|-----|-----------|----------|---------|
| T | T | T | T | F |
| T | F | F | T | F |
| T | U | U | T | F |
| F | U | F | U | T |
| U | U | U | U | U |

Le `U` represente "pas encore decide". La semantique grounded d'un ADF est un **fixed-point 3-valued** : on part de tout-U et on propage jusqu'a stabilisation.


In [1]:
// Prong B (#3801): 0 NuGet, 0 jpype, 0 IKVM, 0 JVM. BCL .NET 9 uniquement.
using System;
using System.Text;
using System.Collections.Generic;
using System.Linq;

// --- Logique 3-valued (Kleene) : T=true, F=false, U=null ---
public enum Tri { T, F, U }

public static class Kleene
{
    public static Tri Not(Tri x) => x == Tri.T ? Tri.F : (x == Tri.F ? Tri.T : Tri.U);
    public static Tri And(Tri a, Tri b)
    {
        if (a == Tri.F || b == Tri.F) return Tri.F;
        if (a == Tri.T && b == Tri.T) return Tri.T;
        return Tri.U;
    }
    public static Tri Or(Tri a, Tri b)
    {
        if (a == Tri.T || b == Tri.T) return Tri.T;
        if (a == Tri.F && b == Tri.F) return Tri.F;
        return Tri.U;
    }
    public static string Str(Tri x) => x == Tri.T ? "t" : (x == Tri.F ? "f" : "u");
}

public static void Show(string label, string s) => $"{label}: {s}".Display();

Show("Kleene", $"Not(T)={Kleene.Str(Kleene.Not(Tri.T))}, And(T,U)={Kleene.Str(Kleene.And(Tri.T,Tri.U))}, Or(F,U)={Kleene.Str(Kleene.Or(Tri.F,Tri.U))}");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Kleene: Not(T)=f, And(T,U)=u, Or(F,U)=u

## 3. Abstract Dialectical Frameworks (ADF)

Un **ADF** (Brewka & Woltran 2010) associe a chaque argument `a` une **condition d'acceptation** `C(a)` : une formule logique sur les autres arguments. `a` est accepte ssi `C(a)` est satisfaite.

### 3.1 Semantique grounded (fixed-point 3-valued)

On part de l'interpretation `I0` ou tout est `U` (indetermine). A chaque tour, on evalue `C(a)` sous l'interpretation courante (en logique 3-valued). Si `C(a)` donne `T`, `a` devient `T` ; si `F`, `a` devient `F` ; si `U`, `a` reste `U`. On itere jusqu'au fixed-point : c'est l'**extension grounded**.

### 3.2 Exemple (verifiable a la main)

Trois arguments avec leurs conditions :
- `a : not b` (a accepte ssi b est faux)
- `c : T` (c toujours accepte - tautologie)
- `b : not c` (b accepte ssi c est faux)

**Re-derivation** : `c = T` (tautologie). Donc `b = not c = not T = F`. Donc `a = not b = not F = T`. Resultat attendu : `{a=T, c=T, b=F}`, extension grounded `{a, c}`.


In [2]:
// --- ADF from-scratch : conditions d'acceptation 3-valued + grounded fixed-point ---
// Une condition d'acceptation = fonction (interpretation -> Tri).
using Interpretation = System.Collections.Generic.Dictionary<string, Tri>;

public sealed class ADF
{
    public List<string> Args { get; } = new();
    public Dictionary<string, Func<Interpretation, Tri>> Conditions { get; } = new();
    public void Add(string a, Func<Interpretation, Tri> cond) { Args.Add(a); Conditions[a] = cond; }

    // Grounded = fixed-point depuis tout-U.
    public Interpretation Grounded()
    {
        var I = new Interpretation();
        foreach (var a in Args) I[a] = Tri.U;
        bool changed = true;
        while (changed)
        {
            changed = false;
            foreach (var a in Args)
            {
                Tri cur = I[a];
                Tri val = Conditions[a](I);   // evalue C(a) sous interpretation courante
                if (val != cur) { I[a] = val; changed = true; }
            }
        }
        return I;
    }
}

// Helpers : references a un argument, constantes T/F
Func<Interpretation, Tri> Var(string x) => I => I.TryGetValue(x, out var v) ? v : Tri.U;
Func<Interpretation, Tri> Const(Tri c) => I => c;
Func<Interpretation, Tri> Neg(Func<Interpretation, Tri> p) => I => Kleene.Not(p(I));
Func<Interpretation, Tri> AndF(Func<Interpretation, Tri> p, Func<Interpretation, Tri> q) => I => Kleene.And(p(I), q(I));
Func<Interpretation, Tri> OrF(Func<Interpretation, Tri> p, Func<Interpretation, Tri> q) => I => Kleene.Or(p(I), q(I));

var adf = new ADF();
adf.Add("a", Neg(Var("b")));   // a : not b
adf.Add("c", Const(Tri.T));    // c : T
adf.Add("b", Neg(Var("c")));   // b : not c

var g = adf.Grounded();
var sb = new StringBuilder();
sb.AppendLine("ADF : a:not(b), c:T, b:not(c)");
sb.AppendLine("  Interpretation grounded :");
foreach (var a in adf.Args)
    sb.AppendLine($"    {a} = {Kleene.Str(g[a])}");
var groundedSet = adf.Args.Where(a => g[a] == Tri.T).ToList();
sb.AppendLine($"  Extension grounded (args True) : {{{string.Join(", ", groundedSet)}}}");
sb.AppendLine();
sb.AppendLine("  Re-derivation : c=T (tautologie) -> b=not c=F -> a=not b=T -> {a,c}.");
Show("ADF", sb.ToString());


ADF: ADF : a:not(b), c:T, b:not(c)
  Interpretation grounded :
    a = t
    c = t
    b = f
  Extension grounded (args True) : {a, c}

  Re-derivation : c=T (tautologie) -> b=not c=F -> a=not b=T -> {a,c}.


### 3.3 Interpretation : pourquoi c'est plus expressif que Dung

Dans un cadre de Dung, l'attaque `c -> b` signifie "c refute b" mais c'est **binaire** : si c est accepte, b est拒e. L'ADF permet des conditions **arbitraires** : `b = not(c) and d`, `a = b or c`, etc. La condition d'acceptation est une formule logique complete, pas juste une liste d'attaquants.

Ici l'exemple est simple mais demontre le mecanisme : `c` est un **fait** (tautologie), `b` est defini comme le reflet de `not c`, et `a` comme `not b`. Le fixed-point converge en une seule iteration complete car il n'y a pas de cycle (c force, b depend de c, a depend de b).


## 4. SetAF : attaques d'ensembles (Nielsen-Parsons 2011)

Dans un **SetAF**, une attaque part d'un **ensemble** d'arguments vers un argument cible : `(B, x)` signifie "l'ensemble B attacks x". L'attaque est **active** ssi **tous** les membres de `B` sont acceptes (IN). C'est plus expressif que Dung : il faut une *coalition* pour attaquer.

### 4.1 Exemple (verifiable a la main)

Arguments `{a, b, c, d}` avec attaques :
- `({b, d}, a)` : l'ensemble {b,d} attaque a
- `({a, c}, c)` : l'ensemble {a,c} attaque c

**Re-derivation du grounded** (labelling IN/OUT/UNDEC, fixed-point) :
1. `b` non attaque -> IN. `d` non attaque -> IN.
2. Attaque `({b,d}, a)` : b et d tous deux IN -> **active** -> `a` OUT.
3. Attaque `({a,c}, c)` : `a` est OUT -> l'ensemble {a,c} n'est jamais complet -> attaque **jamais active** -> `c` IN.
4. Resultat : IN = `{b, c, d}`, OUT = `{a}`.

> **Insight** : l'auto-attaque `({a,c}, c)` echoue car elle exige `a` IN, mais `a` est OUT. La coalition requise ne peut jamais se former.


In [3]:
// --- SetAF from-scratch : attaques d'ensembles + grounded labelling ---
public sealed class SetAF
{
    public List<string> Args { get; } = new();
    // (ensemble attaquants, cible)
    public List<(HashSet<string> Attackers, string Target)> Attacks { get; } = new();
    public SetAF(IEnumerable<string> args) { Args.AddRange(args); }
    public void AddAttack(IEnumerable<string> attackers, string target)
        => Attacks.Add((new HashSet<string>(attackers), target));

    // Labelling grounded : IN si aucune attaque active, OUT si une attaque active, sinon UNDEC.
    public (HashSet<string> IN, HashSet<string> OUT) Grounded()
    {
        var IN = new HashSet<string>();
        var OUT = new HashSet<string>();
        bool changed = true;
        while (changed)
        {
            changed = false;
            foreach (var x in Args)
            {
                if (IN.Contains(x) || OUT.Contains(x)) continue;
                bool defeated = Attacks.Any(atk => atk.Target == x && atk.Attackers.IsSubsetOf(IN));
                if (defeated) { OUT.Add(x); changed = true; continue; }
                // x est IN si toutes les attaques sur x ont au moins un attaquant OUT (attaque "bloquee")
                // ou s'il n'y a aucune attaque sur x.
                var onX = Attacks.Where(atk => atk.Target == x).ToList();
                if (onX.Count == 0 || onX.All(atk => atk.Attackers.Any(b => OUT.Contains(b))))
                {
                    IN.Add(x); changed = true;
                }
            }
        }
        return (IN, OUT);
    }
}

var setaf = new SetAF(new[] { "a", "b", "c", "d" });
setaf.AddAttack(new[] { "b", "d" }, "a");
setaf.AddAttack(new[] { "a", "c" }, "c");
var (sIN, sOUT) = setaf.Grounded();

var sb4 = new StringBuilder();
sb4.AppendLine("SetAF : ({b,d}->a), ({a,c}->c)");
sb4.AppendLine($"  IN  = {{{string.Join(", ", sIN.OrderBy(x=>x))}}}");
sb4.AppendLine($"  OUT = {{{string.Join(", ", sOUT.OrderBy(x=>x))}}}");
sb4.AppendLine($"  Extension grounded = IN = {{{string.Join(", ", sIN.OrderBy(x=>x))}}}");
sb4.AppendLine();
sb4.AppendLine("  Verif : b,d non attaquants -> IN ; ({b,d},a) active -> a OUT ;");
sb4.AppendLine("          ({a,c},c) requiert a IN mais a OUT -> jamais active -> c IN.");
Show("SetAF", sb4.ToString());


SetAF: SetAF : ({b,d}->a), ({a,c}->c)
  IN  = {b, c, d}
  OUT = {a}
  Extension grounded = IN = {b, c, d}

  Verif : b,d non attaquants -> IN ; ({b,d},a) active -> a OUT ;
          ({a,c},c) requiert a IN mais a OUT -> jamais active -> c IN.


### 4.2 Pourquoi {a,c} n'attaque jamais c

L'attaque `({a,c}, c)` est **auto-referentielle** : pour qu'elle soit active, il faudrait `a` ET `c` tous deux IN. Mais `c` IN est precisement ce que cette attaque empeche, et `a` est OUT (defait par {b,d}). La coalition requise `{a,c}` ne peut donc jamais se former : l'attaque est structurellement inactive. C'est une subtilite que la semantique ensemble capture naturellement, la ou une lecture "il existe un attaquant" (Dung binaire) induirait en erreur.


## 5. EAF : attaques sur les attaques (Modgil 2010)

Dans un **EAF** (Extended Argumentation Framework), une attaque peut porter sur une **autre attaque**, pas seulement sur un argument. Si `c` attaque l'attaque `(a, b)` (notation `c -> (a,b)`), et que `c` est accepte, alors l'attaque `a -> b` est **revoquee** : `b` est reinstated.

### 5.1 Exemple (verifiable a la main)

Arguments `{a, b, c}` :
- Attaque standard `a -> b` (a attaque b)
- **Meta-attaque** `c -> (a, b)` (c attaque l'attaque de a vers b)
- `a` et `c` non attaques.

**Sans la meta-attaque (Dung classique)** : `a -> b` defait `b`. Grounded `{a, c}` (b OUT).

**Avec la meta-attaque (EAF)** : `c` est IN (non attaque), donc `c -> (a,b)` revoque l'attaque `a -> b`. Cette attaque n'est plus un **defeat** valide. Donc `b` n'est defait par personne -> `b` IN. Grounded `{a, b, c}`.

> **Lecon (Modgil)** : une meta-attaque depuis un argument accepte **desactive** l'attaque cible, restaurant l'argument qu'elle menacait. C'est le mecanisme formel des **preferences** et des **exceptions** en argumentation.


In [4]:
// --- EAF from-scratch : meta-attaques (Modgil) + reinstatement ---
public sealed class EAF
{
    public List<string> Args { get; } = new();
    public List<(string Src, string Dst)> Attacks { get; } = new();              // a -> b
    public List<(string Src, string SrcAtk, string DstAtk)> MetaAttacks { get; } = new(); // c -> (a,b)

    public EAF(IEnumerable<string> args) { Args.AddRange(args); }
    public void AddAttack(string src, string dst) => Attacks.Add((src, dst));
    public void AddMetaAttack(string src, string aSrc, string aDst) => MetaAttacks.Add((src, aSrc, aDst));

    // Un attack (a,b) est un DEFEAT valide ssi aucun argument accepte ne le meta-attaque.
    bool IsValid((string Src, string Dst) at, HashSet<string> IN, bool useMeta)
    {
        if (!useMeta) return true;   // Dung classique : toute attaque compte
        // attaque revoquee ssi un meta-attaquant est accepte (IN)
        return !MetaAttacks.Any(m => m.SrcAtk == at.Src && m.DstAtk == at.Dst && IN.Contains(m.Src));
    }

    // Grounded = least fixed-point de la fonction caracteristique F(S) = {x : validDefeats(x) subset OUT(S)},
    // ou OUT(S) = args defaits par un membre de S. On recalcule l'ensemble IN complet a chaque tour
    // (pas de verrouillage d'etiquette) pour que les meta-attaques activables en cours d'iteration
    // puissent reinstaurer un argument (sinon ordre de traitement = faux negatif).
    public HashSet<string> Grounded(bool useMeta)
    {
        var IN = new HashSet<string>();
        for (int iter = 0; iter < 200; iter++)
        {
            // defeaters valides de chaque x selon l'IN courant
            var vd = Args.ToDictionary(x => x, x => Attacks
                .Where(at => at.Dst == x && IsValid(at, IN, useMeta))
                .Select(at => at.Src).Distinct().ToList());
            var OUT = new HashSet<string>(Args.Where(x => vd[x].Any(y => IN.Contains(y))));
            // F(S) : x IN ssi tous ses defeaters valides sont dans OUT (vacuous si aucun defeater)
            var newIN = new HashSet<string>(Args.Where(x => vd[x].All(y => OUT.Contains(y))));
            if (newIN.SetEquals(IN)) break;
            IN = newIN;
        }
        return IN;
    }
}

// Exemple EAF : a->b, c->(a,b), a et c non attaques
var eaf = new EAF(new[] { "a", "b", "c" });
eaf.AddAttack("a", "b");
eaf.AddMetaAttack("c", "a", "b");

var dungG = eaf.Grounded(useMeta: false);
var eafG  = eaf.Grounded(useMeta: true);

var sb5 = new StringBuilder();
sb5.AppendLine("EAF : a->b, c->(a,b) [c meta-attaque l'attaque a->b]");
sb5.AppendLine($"  Dung classique (sans meta)   : {{{string.Join(", ", dungG.OrderBy(x=>x))}}}");
sb5.AppendLine($"  EAF (avec meta-attaque c)    : {{{string.Join(", ", eafG.OrderBy(x=>x))}}}");
sb5.AppendLine();
sb5.AppendLine("  Verif EAF : c IN (non attaque) -> c revoque a->b -> b non defait -> b IN.");
sb5.AppendLine("              La meta-attaque REINSTATED b (Dung {a,c} -> EAF {a,b,c}).");
Show("EAF", sb5.ToString());


EAF: EAF : a->b, c->(a,b) [c meta-attaque l'attaque a->b]
  Dung classique (sans meta)   : {a, c}
  EAF (avec meta-attaque c)    : {a, b, c}

  Verif EAF : c IN (non attaque) -> c revoque a->b -> b non defait -> b IN.
              La meta-attaque REINSTATED b (Dung {a,c} -> EAF {a,b,c}).


### 5.2 Interpretation : preferences et exceptions

La meta-attaque `c -> (a,b)` encode une **preference** : "l'argument c (peut-etre un argument de plus haut niveau) dit que la critique de a contre b n'est pas valable". C'est exactement le mecanisme des **frameworks avec preferences** (Modgil 2010) : au lieu de supprimer l'attaque `a->b`, on permet a un argument de la **desactiver** lorsqu'il est accepte. Cela rend l'argumentation **dynamique** et **hiérarchique** : on peut argumenter *contre* une objection, pas seulement contre une conclusion.


## 6. VAF : argumentation value-based (Bench-Capon 2003)

Dans un **VAF**, chaque argument promeut une **valeur** (ex. "economie", "environnement", "justice"). Une attaque `a -> b` devient un **defeat** ssi la valeur de `a` est **preferée** (ou egale) a celle de `b` dans un ordre de valeurs `ValPref`. Sinon, l'attaque **echoue** : `b` survive.

### 6.1 Exemple (verifiable a la main)

Arguments :
- `a` promeut la valeur **Economie**, `b` promeut **Environnement**.
- Attaque `a -> b`.

Cas 1 : `Economie > Environnement` -> `a` defait `b` -> grounded `{a}`.
Cas 2 : `Environnement > Economie` -> l'attaque `a->b` echoue (b prioritaire) -> `b` non defait -> grounded `{a, b}`.

> **Lecon (Bench-Capon)** : la meme topologie d'attaque donne des **extensions differentes** selon l'ordre des valeurs. Le VAF formalise l'idee qu'un argument "plus fort" (valeur prioritaire) peut resister a une attaque.


In [5]:
// --- VAF from-scratch : value-based (Bench-Capon) ---
public sealed class VAF
{
    public List<string> Args { get; } = new();
    public Dictionary<string, string> ValueOf { get; } = new();  // arg -> valeur
    public List<(string Src, string Dst)> Attacks { get; } = new();
    public Dictionary<(string, string), int> ValuePref { get; } = new(); // (vA, vB) -> +1 si vA prefere a vB

    public void Add(string arg, string value) { Args.Add(arg); ValueOf[arg] = value; }
    public void AddAttack(string src, string dst) => Attacks.Add((src, dst));
    // Definit un ordre : v1 preferee a v2
    public void SetPref(string v1, string v2) { ValuePref[(v1, v2)] = 1; ValuePref[(v2, v1)] = -1; }

    // Attack valide (defeat) ssi value(src) preferee ou egale a value(dst).
    bool IsValid((string Src, string Dst) at)
    {
        string vy = ValueOf[at.Src], vx = ValueOf[at.Dst];
        if (vy == vx) return true;                                   // meme valeur -> attaque reussit
        if (ValuePref.TryGetValue((vy, vx), out int p)) return p >= 0; // vy pref a vx
        return false;                                                // pas de pref -> attaque echoue
    }

    public HashSet<string> Grounded()
    {
        var IN = new HashSet<string>();
        var OUT = new HashSet<string>();
        bool changed = true;
        while (changed)
        {
            changed = false;
            foreach (var x in Args)
            {
                if (IN.Contains(x) || OUT.Contains(x)) continue;
                var validAtkrs = Attacks
                    .Where(at => at.Dst == x && IsValid(at))
                    .Select(at => at.Src).Distinct().ToList();
                if (validAtkrs.Any(y => IN.Contains(y))) { OUT.Add(x); changed = true; continue; }
                if (validAtkrs.Count == 0 || validAtkrs.All(y => OUT.Contains(y)))
                {
                    IN.Add(x); changed = true;
                }
            }
        }
        return IN;
    }
}

VAF RunVAF(string prefDesc, Action<VAF> setPref)
{
    var vaf = new VAF();
    vaf.Add("a", "Economie");
    vaf.Add("b", "Environnement");
    vaf.AddAttack("a", "b");
    setPref(vaf);
    var g = vaf.Grounded();
    var sb = new StringBuilder();
    sb.AppendLine($"VAF (a:Economie -> b:Environnement), {prefDesc}");
    sb.AppendLine($"  Extension grounded : {{{string.Join(", ", g.OrderBy(x=>x))}}}");
    Show("VAF", sb.ToString());
    return vaf;
}

RunVAF("Economie > Environnement", v => v.SetPref("Economie", "Environnement"));
RunVAF("Environnement > Economie", v => v.SetPref("Environnement", "Economie"));


VAF: VAF (a:Economie -> b:Environnement), Economie > Environnement
  Extension grounded : {a}


VAF: VAF (a:Economie -> b:Environnement), Environnement > Economie
  Extension grounded : {a, b}


### 6.2 Interpretation : le defait conditionnel

Dans le cas 1 (Economie prioritaire), `a` defait `b` car sa valeur l'emporte. Dans le cas 2 (Environnement prioritaire), `b` resiste : son attaque depuis `a` echoue car la valeur d'`a` est dominée. La topologie `a -> b` est **identique**, mais l'extension grounded change. C'est la puissance du VAF : **l'ordre de valeurs est un parametre du raisonnement**, pas juste une etiquette.

> **Lien avec EAF** : le VAF est implementable comme un EAF ou les meta-attaques correspondent aux preferences de valeurs. Les deux formalismes capturent l'idee que **toutes les attaques ne sont pas des defeats**.


## 7. Synthese : 4 extensions, 4 sources d'expressivite

| Framework | Source d'expressivite | Exemple verifie | Grounded |
|-----------|----------------------|-----------------|----------|
| **Dung (base)** | attaque binaire | - | - |
| **ADF** | condition d'acceptation 3-valued | `a:not b, c:T, b:not c` | `{a, c}` |
| **SetAF** | attaque d'ensemble (coalition requise) | `{b,d}->a, {a,c}->c` | `{b, c, d}` |
| **EAF** | meta-attaque (attaque d'attaque) | `a->b, c->(a,b)` | `{a, b, c}` (vs Dung `{a,c}`) |
| **VAF** | ordre de valeurs | `a:Eco -> b:Env` | `{a}` ou `{a,b}` selon pref |

### Ce que nous avons appris

1. **ADF** : generaliser l'attaque en **formule logique** sur les arguments (logique 3-valued de Kleene pour le grounded).
2. **SetAF** : une attaque peut necessiter une **coalition** complete ; l'auto-reference la desactive naturellement.
3. **EAF** : un argument peut **desactiver une attaque** (preference/exception), restaurant sa cible.
4. **VAF** : la defaite est **conditionnelle** a un ordre de valeurs ; meme graphe, extensions differentes.

### Value-add du jumeau (Prong B, #3801)

- **0 jpype, 0 JVM, 0 IKVM** : BCL .NET 9 uniquement, la ou le twin Python depend de la JVM Tweety + NativeMinisat.
- **Semantiques observables** : conditions d'acceptation comme `Func<Interpretation,Tri>`, labelling grounded explicite, meta-attaques comme liste `(c,(a,b))`.
- **4 piliers verifies** : chaque extension grounded re-derivable a la main.

---

**Notebook suivant** : [Tweety-7b-Ranking-Probabilistic](Tweety-7b-Ranking-Probabilistic.ipynb) | [Retour au sommaire](README.md)


## 8. Exercices

### Exercice 1 : ADF cyclique

Construisez un ADF avec `a : not b` et `b : not a` (cycle). Que donne le grounded ? (Indice : en logique 3-valued, aucun ne peut etre decide -> tous U -> extension grounded vide.)

### Exercice 2 : SetAF a 3 arguments

Arguments `{x, y, z}`, attaques `({x,y}, z)` et `({z}, x)`. Calculez le grounded. (Indice : qui est IN ? qui est OUT ?)

### Exercice 3 : EAF avec meta-attaque echouant

Reprenez l'exemple EAF (`a->b, c->(a,b)`) mais ajoutez une attaque `d -> c` avec `d` non attaque. Desormais `c` est OUT. Que devient le grounded ? (Indice : `c` OUT -> meta-attaque inactive -> `a->b` redevient un defeat -> `b` OUT.)

### Exercice 4 : VAF a 3 valeurs

Arguments `a` (Economie), `b` (Environnement), `c` (Justice), attaques `a->b`, `b->c`, `c->a` (cycle). Testez les 3 ordres cycliques et identifiez celui qui donne le plus grand grounded.


In [6]:
// Exercice 1 : ADF cyclique (a:not b, b:not a)
// TODO etudiant : construire l'ADF et observer le grounded
var adfExo1 = new ADF();
// adfExo1.Add("a", Neg(Var("b")));
// adfExo1.Add("b", Neg(Var("a")));
// var gExo1 = adfExo1.Grounded();
Show("Exo1", "Exercice a completer : ADF cyclique (attendu : extension grounded vide, tous U)");


Exo1: Exercice a completer : ADF cyclique (attendu : extension grounded vide, tous U)

In [7]:
// Exercice 2 : SetAF ({x,y}->z, {z}->x)
// TODO etudiant : construire le SetAF et calculer le grounded
var setafExo2 = new SetAF(new[] { "x", "y", "z" });
// setafExo2.AddAttack(new[]{"x","y"}, "z");
// setafExo2.AddAttack(new[]{"z"}, "x");
// var (inExo2, outExo2) = setafExo2.Grounded();
Show("Exo2", "Exercice a completer : SetAF 3 args (qui est IN ?)");


Exo2: Exercice a completer : SetAF 3 args (qui est IN ?)

In [8]:
// Exercice 3 : EAF avec c defait (d->c ajoutee)
// TODO etudiant : ajouter d et d->c, observer le grounded
var eafExo3 = new EAF(new[] { "a", "b", "c", "d" });
// eafExo3.AddAttack("a","b"); eafExo3.AddMetaAttack("c","a","b"); eafExo3.AddAttack("d","c");
// var gExo3 = eafExo3.Grounded(useMeta: true);
Show("Exo3", "Exercice a completer : EAF avec c OUT -> b OUT (reinstatement perdu)");


Exo3: Exercice a completer : EAF avec c OUT -> b OUT (reinstatement perdu)

In [9]:
// Exercice 4 : VAF a 3 valeurs cyclique
// TODO etudiant : cycle a->b->c->a avec 3 valeurs, tester les ordres
var vafExo4 = new VAF();
// vafExo4.Add("a","Economie"); vafExo4.Add("b","Environnement"); vafExo4.Add("c","Justice");
// vafExo4.AddAttack("a","b"); vafExo4.AddAttack("b","c"); vafExo4.AddAttack("c","a");
Show("Exo4", "Exercice a completer : VAF 3-valeurs cyclique (quel ordre maximise le grounded ?)");


Exo4: Exercice a completer : VAF 3-valeurs cyclique (quel ordre maximise le grounded ?)

## Conclusion

Ce jumeau C# a presente les **4 extensions principales** du cadre de Dung, reimplementation from-scratch du twin Python jpype/Tweety JVM.

### Resultats cles (Math Verification Standard)

| Pilier | Entree | Sortie (re-derivable a la main) |
|--------|--------|--------------------------------|
| **ADF** | `a:not b, c:T, b:not c` | grounded `{a,c}` (c=T -> b=F -> a=T) |
| **SetAF** | `{b,d}->a, {a,c}->c` | grounded `{b,c,d}` (coalition {a,c} jamais formee) |
| **EAF** | `a->b, c->(a,b)` | grounded `{a,b,c}` (meta-attaque reinstated b) |
| **VAF** | `a:Eco -> b:Env` | `{a}` si Eco>Env, `{a,b}` si Env>Eco |

### Value-add (Prong B, #3801)

- **0 jpype/JVM/IKVM** : BCL .NET 9 uniquement, la ou le twin Python charge 42 JARs + NativeMinisat.
- **Semantiques from-scratch** : Kleene 3-valued, labelling grounded, meta-attaques, ordre de valeurs.

**Suite** : [Tweety-7b-Ranking-Probabilistic](Tweety-7b-Ranking-Probabilistic.ipynb) | [Retour au sommaire](README.md)
